# Making SIMSOPT GPU native: NVIDIA profile

Before running any cell, select **Runtime > Change runtime type > GPU**. This notebook refuses to profile if JAX falls back to the CPU. It clones the public feature branch, builds SIMSOPT, runs CPU/GPU parity checks, separates compilation from steady-state timing, and exports a Perfetto trace plus a device-memory profile.

In [ ]:
import subprocess

subprocess.run(["nvidia-smi"], check=True)

In [ ]:
import os
import sys
from pathlib import Path

repo = Path("/content/simsopt")
if not repo.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            "gpu-native-objective",
            "https://github.com/PedroFranciscoGil/simsopt.git",
            str(repo),
        ],
        check=True,
    )
else:
    subprocess.run(
        ["git", "fetch", "origin", "gpu-native-objective"], cwd=repo, check=True
    )
    subprocess.run(["git", "switch", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".", "pytest"], cwd=repo, check=True
)
os.chdir(repo)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

In [ ]:
import jax
from simsopt.gpu import backend_report

report = backend_report()
print(report)
assert jax.default_backend() == "gpu", report
assert any(device.platform == "gpu" for device in jax.devices()), report

In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/gpu/test_biot_savart.py",
        "tests/gpu/test_objective.py",
        "tests/gpu/test_adapters.py",
    ],
    check=True,
)

In [ ]:
artifact_root = Path("/content/simsopt-gpu-profile")
result_file = artifact_root / "minimal-gpu.json"
trace_dir = artifact_root / "trace"
memory_file = artifact_root / "device-memory.prof"

subprocess.run(
    [
        sys.executable,
        "benchmarks/gpu/profile_gpu_objective.py",
        "--problem",
        "minimal",
        "--warmup",
        "3",
        "--repeats",
        "10",
        "--target-tile-size",
        "128",
        "--source-tile-size",
        "256",
        "--trace-dir",
        str(trace_dir),
        "--memory-profile",
        str(memory_file),
        "--output",
        str(result_file),
    ],
    check=True,
)

In [ ]:
import json

results = json.loads(result_file.read_text())
print(json.dumps(results, indent=2))

assert results["environment"]["jax_backend"] == "gpu"
assert results["parity"]["value_absolute_error"] < 1e-9
assert results["parity"]["curve_gradient_relative_l2_error"] < 1e-7

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive("/content/simsopt-gpu-profile", "zip", artifact_root)
files.download(archive)